In [1]:
print("start")

start


In [1]:
%pwd

'd:\\RAG_PROJ\\Dynamic-Knowledge-Retrieval-Chatbot-RAG-\\research'

In [2]:
import os
os.chdir("../")

In [4]:
%pwd

'd:\\RAG_PROJ\\Dynamic-Knowledge-Retrieval-Chatbot-RAG-'

In [4]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

d:\SNAKE\envs\chatbot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def load_all_data(data,faq_data=None):
    loader=DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents=loader.load()
    if not documents:
        print(" No PDF files found in the directory.")
    else:
        print(f"Loaded {len(documents)} documents")

    faq_documents= []
    if faq_data:
        for item in faq_data:
            text = f"Question: {item['q']}\nAnswer: {item['a']}"
            faq_documents.append(
                Document(
                    page_content=text,
                    metadata={"source": "FAQ"}
                )
            )

        print(f" Loaded {len(faq_documents)} FAQ entries")


    all_documents =documents + faq_documents

    print(f"Total documents: {len(all_documents)}")

    return all_documents

In [6]:
faq_data=[ {"q": "What is arrhythmia?", "a": "Irregular heartbeat condition."},
    {"q": "What are symptoms?", "a": "Dizziness, chest pain, palpitations."},
    {"q": "What causes arrhythmia?", "a": "Heart disease, stress, BP."}]
documents = load_all_data("data", faq_data)

Loaded 67 documents
 Loaded 3 FAQ entries
Total documents: 70


In [8]:
print(documents)

[Document(metadata={'producer': 'Acrobat Distiller 19.0 (Windows)', 'creator': 'PyPDF', 'creationdate': '2020-01-20T15:54:03+00:00', 'moddate': '2023-10-16T14:17:52+01:00', 'title': 'untitled', 'source': 'data\\Arrhythmias.pdf', 'total_pages': 40, 'page': 0, 'page_label': '1'}, page_content='Arrhythmias: Understanding \nyour condition\nRegistered Charity No. 1107496\nwww.heartrhythmalliance.org\nWorking together to improve the diagnosis, treatment \nand quality of life for all those aﬀ ected by arrhythmias'), Document(metadata={'producer': 'Acrobat Distiller 19.0 (Windows)', 'creator': 'PyPDF', 'creationdate': '2020-01-20T15:54:03+00:00', 'moddate': '2023-10-16T14:17:52+01:00', 'title': 'untitled', 'source': 'data\\Arrhythmias.pdf', 'total_pages': 40, 'page': 1, 'page_label': '2'}, page_content='2\nGlossary\nArrhythmia  An abnormal heart rhythm\nAtrium  Top chambers of the heart that receive \nblood from the body and from the lungs. The right \natrium is where the heart’\ns natural pac

In [9]:
len(documents)

70

In [7]:
from typing import List

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs: List[Document]=[]
    for doc in docs:
        src=doc.metadata.get("source")
        minimal_docs.append(
            Document(page_content=doc.page_content,
                     metadata={"source": src})
        )
    return minimal_docs

In [8]:
minimal_docs=filter_to_minimal_docs(documents)

In [12]:
minimal_docs

[Document(metadata={'source': 'data\\Arrhythmias.pdf'}, page_content='Arrhythmias: Understanding \nyour condition\nRegistered Charity No. 1107496\nwww.heartrhythmalliance.org\nWorking together to improve the diagnosis, treatment \nand quality of life for all those aﬀ ected by arrhythmias'),
 Document(metadata={'source': 'data\\Arrhythmias.pdf'}, page_content='2\nGlossary\nArrhythmia  An abnormal heart rhythm\nAtrium  Top chambers of the heart that receive \nblood from the body and from the lungs. The right \natrium is where the heart’\ns natural pacemaker \n(sino atrial node) can be found\nAsystole  Cessation of heartbeat\nBradycardia  A slow heart rate, normally less than \n60 beats per minute\nCardiac arrest  A sudden cessation of the heart \nto contract eﬀ ectively\n, or at all\nCardiologist  A doctor who has specialised in \nthe diagnosis and treatment of patients with a \nheart condition\nCardioversion  A procedure by which an abnormally \nfast heart rate (tachycardia) or other ca

In [9]:
# split docs into chunks
def text_split(minimal_docs):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    text_chunk=text_splitter.split_documents(minimal_docs)
    return text_chunk

In [10]:
text_chunk=text_split(minimal_docs)
print(f"number of chunks:{len(text_chunk)}")

number of chunks:244


In [11]:
from langchain.embeddings import HuggingFaceEmbeddings
def download_embeddings():
    model_name="sentence-transformers/all-MiniLM-L6-v2"
    embeddings=HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings
embedding=download_embeddings()

C:\Users\NAVEEN KUMAR\AppData\Local\Temp\ipykernel_17028\3562430573.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(


In [12]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [13]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [14]:
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")
os.environ["PINECONE_API_KEY"]=PINECONE_API_KEY
os.environ["OPENAI_API_KEY"]=OPENAI_API_KEY

In [15]:
print(OPENAI_API_KEY)

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


In [16]:
from pinecone import Pinecone
pinecode_api_key=PINECONE_API_KEY
pc=Pinecone(api_key=pinecode_api_key)

In [17]:
from pinecone import ServerlessSpec
index_name="chatbot"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384, #dimension of embeddings
        metric="cosine",
        spec=ServerlessSpec(cloud='aws',region="us-east-1")
    )
index=pc.Index(index_name)

In [18]:
from langchain_pinecone import PineconeVectorStore
docsearch=PineconeVectorStore.from_documents(
    documents=text_chunk,
    embedding=embedding,
    index_name=index_name
)

In [36]:
retriever=docsearch.as_retriever(search_type="similarity",search_kwargs={"k":4})

In [20]:
retrieved_docs=retriever.invoke("what is arrhthymia")
retrieved_docs

[Document(id='8cb19114-a804-4aea-8e18-be260a994cc2', metadata={'source': 'data\\Arrhythmias.pdf'}, page_content='What are arrhythmias?\n5\nArrhythmias are disorders of your heart’s electrical system whereby there is a \nchange in the regular beat of your heart. Sometimes if the conduction pathway \nis damaged or becomes blocked, or if an extra pathway exists, the heart’s \nrhythm changes. The heart may beat too quickly (tachycardia), too slowly \n(bradycardia) or irregularly which may aﬀ ect the heart’s ability to eﬀ ectively \npump blood around the body. These abnormal heart rhythms are known'),
 Document(id='ac89030a-a25d-4f06-83ed-ccebb630bb19', metadata={'source': 'data\\Arrhythmias.pdf'}, page_content='What are arrhythmias?\n5\nArrhythmias are disorders of your heart’s electrical system whereby there is a \nchange in the regular beat of your heart. Sometimes if the conduction pathway \nis damaged or becomes blocked, or if an extra pathway exists, the heart’s \nrhythm changes. The 

In [26]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openrouter/free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0,
    max_tokens=150
)

In [22]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [33]:
system_prompt = """
You are an AI assistant for ECG and arrhythmia analysis.
Answer using the provided context as the primary source.
If the answer is partially available, explain based on context.

If the answer is completely not found, say:
"I don't have enough information."
Keep answers clear and concise.
"""
prompt=ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human", "Context:\n{context}\n\nQuestion:\n{input}")
    ]
)

In [28]:
question_answer_chain=create_stuff_documents_chain(llm,prompt)
rag_chain=create_retrieval_chain(retriever,question_answer_chain)

In [40]:
import time
response=rag_chain.invoke({"input":"What can trigger an arrhythmia"})
print(response["answer"])
time.sleep(3)

Common triggers for arrhythmias include stress, caffeine, tobacco, alcohol, diet pills, and cough or cold medicines. However, there is usually an underlying physical reason for arrhythmias, such as electrical variations people are born with or damaged heart tissue from acquired heart disease like myocardial infarction.
